# 🧠 โปรเจกต์ NN: Neural Network Architecture & Training
**เป้าหมาย:** สร้างและสอนโมเดลโครงข่ายประสาทเทียมแบบ **Multilayer Perceptron (MLP)** เพื่อแยกแยะประเภทโรคการนอนหลับ (Classification)
- **Input Layer:** รับข้อมูลพฤติกรรมและสเปคสุขภาพ
- **Hidden Layers:** ชั้นประมวลผลซ่อนเร้นที่ใช้ดึงรูปแบบความสัมพันธ์ที่ซับซ้อน (ใช้ `ReLU` activation)
- **Output Layer:** ทำนายผลลัพธ์ 3 คลาส (0 = ปกติ, 1 = นอนไม่หลับ, 2 = หยุดหายใจขณะหลับ) ด้วย `Softmax` (ใน Scikit-learn จะจัดการให้อัตโนมัติในโหมด Classification)

## 1) Import Libraries (นำเข้าเครื่องมือที่จำเป็น)
โหลดไลบรารีพื้นฐานสำหรับการจัดการตารางข้อมูล (Pandas) และการบันทึกไฟล์ (Joblib)

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

print("✅ Import Libraries สำหรับ Neural Network เรียบร้อย!")

✅ Import Libraries สำหรับ Neural Network เรียบร้อย!


## 2) โหลดข้อมูลและแบ่งชุดข้อมูล (Load & Split Data)
โหลดไฟล์ `sleep_health_cleaned.csv` และแบ่งข้อมูลเป็น Training Set (80%) สำหรับให้โมเดลเรียนรู้ และ Testing Set (20%) สำหรับวัดผล

In [2]:
# 1. โหลดข้อมูลที่ผ่านการ Clean และ Encode มาแล้ว
df = pd.read_csv('../data/sleep_health_cleaned.csv')

# 2. แยก Features (X) และ Target (y)
X = df.drop(columns=['Sleep Disorder'])
y = df['Sleep Disorder']

# 3. แบ่งชุดข้อมูล Train 80% / Test 20%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"📊 จำนวนข้อมูลสำหรับสอนโมเดล (Train): {X_train.shape[0]} แถว")
print(f"📊 จำนวนข้อมูลสำหรับทดสอบ (Test): {X_test.shape[0]} แถว")

📊 จำนวนข้อมูลสำหรับสอนโมเดล (Train): 299 แถว
📊 จำนวนข้อมูลสำหรับทดสอบ (Test): 75 แถว


## 3) การปรับสเกลข้อมูล (Critical Feature Scaling for NN)
**คำเตือน:** โมเดล Neural Network จะทำงานผิดพลาดอย่างรุนแรง (หรือใช้เวลาเทรนนานมาก) หากไม่ปรับสเกลข้อมูล เนื่องจากใช้สมการคณิตศาสตร์ที่อิงกับน้ำหนัก (Weights) 
ดังนั้น การใช้ `StandardScaler` เพื่อปรับให้ข้อมูลมีค่าเฉลี่ย=0 และส่วนเบี่ยงเบนมาตรฐาน=1 จึงเป็นขั้นตอนที่ **ขาดไม่ได้**

In [3]:
# 1. สร้างตัวปรับสเกล
scaler = StandardScaler()

# 2. เรียนรู้สเกลจากชุด Train และปรับค่า
X_train_scaled = scaler.fit_transform(X_train)

# 3. นำไม้บรรทัดมาปรับสเกลชุด Test
X_test_scaled = scaler.transform(X_test)

# 4. บันทึก Scaler เก็บไว้ใช้แปลงข้อมูลจากผู้ใช้หน้าเว็บ
os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/nn_scaler.pkl')

print("💾 บันทึก 'nn_scaler.pkl' เรียบร้อย! (หัวใจสำคัญก่อนโยนข้อมูลเข้า NN หน้าเว็บ)")

💾 บันทึก 'nn_scaler.pkl' เรียบร้อย! (หัวใจสำคัญก่อนโยนข้อมูลเข้า NN หน้าเว็บ)


## 4) ออกแบบและสอนโมเดล Neural Network (Build & Train MLP)
กำหนดสถาปัตยกรรม (Architecture) ของโครงข่ายประสาทเทียม:
- `hidden_layer_sizes=(64, 32)`: มี Hidden Layer 2 ชั้น ชั้นแรกมี 64 โหนด ชั้นที่สองมี 32 โหนด
- `activation='relu'`: ใช้ฟังก์ชัน ReLU ช่วยให้โมเดลเรียนรู้ความสัมพันธ์แบบไม่เป็นเส้นตรง (Non-linear) ได้ดี
- `early_stopping=True`: ระบบหยุดเรียนรู้อัตโนมัติหากพบว่าเทรนต่อไปก็ไม่ฉลาดขึ้น (ป้องกัน Overfitting)

In [4]:
# 1. กำหนดโครงสร้าง Neural Network
nn_model = MLPClassifier(
    hidden_layer_sizes=(64, 32), # จำนวนโหนดในแต่ละชั้น
    activation='relu',           # ฟังก์ชันกระตุ้นยอดนิยม
    solver='adam',               # อัลกอริทึมในการปรับน้ำหนัก (Optimization)
    max_iter=500,                # จำนวนรอบ (Epochs) สูงสุดที่อนุญาตให้เรียนรู้
    early_stopping=True,         # หยุดอัตโนมัติถ้าเริ่ม Overfit
    random_state=42
)

# 2. สั่งให้โครงข่ายประสาทเทียมเริ่มเรียนรู้
print("🧠 Neural Network กำลังเรียนรู้และปรับน้ำหนัก (Synapses)...")
nn_model.fit(X_train_scaled, y_train)

print(f"✅ เทรนเสร็จสมบูรณ์! (ใช้เวลาไปทั้งหมด {nn_model.n_iter_} รอบจาก 500 รอบ)")

🧠 Neural Network กำลังเรียนรู้และปรับน้ำหนัก (Synapses)...
✅ เทรนเสร็จสมบูรณ์! (ใช้เวลาไปทั้งหมด 18 รอบจาก 500 รอบ)


## 5) ประเมินผลและบันทึกโมเดล (Evaluation & Export)
เนื่องจากเป็นโจทย์ประเภทแบ่งกลุ่ม (Classification) เราจะวัดผลด้วย:
- **Accuracy:** ความแม่นยำรวม (ทายถูกกี่เปอร์เซ็นต์)
- **Classification Report:** ดูรายละเอียด Precision, Recall, F1-Score ในแต่ละคลาสโรค

In [5]:
# 1. ให้โมเดลลองทำนายข้อสอบ
y_pred = nn_model.predict(X_test_scaled)

# 2. คำนวณความแม่นยำรวม
accuracy = accuracy_score(y_test, y_pred)

print("🎯 === ผลการประเมินความแม่นยำของ Neural Network ===")
print(f"Accuracy: {accuracy * 100:.2f}%\n")

print("📋 รายละเอียดแต่ละคลาส (Classification Report):")
# target_names ต้องเรียงตามที่ LabelEncoder บันทึกไว้ตอน Clean ข้อมูล
print(classification_report(y_test, y_pred, target_names=['None', 'Insomnia', 'Sleep Apnea']))

# 3. บันทึกตัวโมเดลเพื่อนำไปทำ Web App
joblib.dump(nn_model, '../models/nn_mlp_model.pkl')
joblib.dump({"accuracy": accuracy}, '../models/nn_metrics.pkl')

print("💾 บันทึก 'nn_mlp_model.pkl' และ 'nn_metrics.pkl' เรียบร้อย!")
print("🎉 หลังบ้านโปรเจกต์ Neural Network ปิดจ๊อบอย่างสวยงาม!")

🎯 === ผลการประเมินความแม่นยำของ Neural Network ===
Accuracy: 88.00%

📋 รายละเอียดแต่ละคลาส (Classification Report):
              precision    recall  f1-score   support

        None       0.81      0.81      0.81        16
    Insomnia       0.91      0.95      0.93        43
 Sleep Apnea       0.86      0.75      0.80        16

    accuracy                           0.88        75
   macro avg       0.86      0.84      0.85        75
weighted avg       0.88      0.88      0.88        75

💾 บันทึก 'nn_mlp_model.pkl' และ 'nn_metrics.pkl' เรียบร้อย!
🎉 หลังบ้านโปรเจกต์ Neural Network ปิดจ๊อบอย่างสวยงาม!
